In [4]:
# ============================
# Mini BPE Learner (Short + Simple)
# ============================

from collections import Counter

# End-of-word marker (used to mark word boundaries)
EOW = "_"


# ----------------------------
# Convert a word into characters + '_'
# Example: "new" -> ['n','e','w','_']
# ----------------------------
def tok_word(word):
    return list(word) + [EOW]


# ----------------------------
# Convert full corpus into tokenized words
# ----------------------------
def tok_corpus(text):
    return [tok_word(w) for w in text.split()]


# ----------------------------
# Count adjacent symbol pairs (bigrams)
# ----------------------------
def bigrams(words):
    counts = Counter()
    for w in words:
        counts.update(zip(w, w[1:]))  # (w[i], w[i+1])
    return counts


# ----------------------------
# Merge a bigram inside ONE word
# Example: ('e','r') -> 'er'
# ----------------------------
def merge_word(word, pair):
    a, b = pair
    merged, i = [], 0
    while i < len(word):
        if i < len(word) - 1 and word[i] == a and word[i+1] == b:
            merged.append(a + b)  # merge the pair
            i += 2
        else:
            merged.append(word[i])
            i += 1
    return merged


# ----------------------------
# Apply merge to ALL words
# ----------------------------
def apply_merge(words, pair):
    return [merge_word(w, pair) for w in words]


# ----------------------------
# Compute current vocabulary size
# ----------------------------
def vocab_size(words):
    return len({token for w in words for token in w})


# ----------------------------
# Learn BPE merges
# ----------------------------
def learn_bpe(text, max_merges=30):
    words = tok_corpus(text)
    learned_merges = []

    for step in range(1, max_merges + 1):
        bg = bigrams(words)

        # Stop if no pairs left
        if not bg:
            print(f"Step {step:02d}: No bigrams left to merge.")
            break

        # Pick most frequent pair
        pair, count = bg.most_common(1)[0]

        # Apply merge
        words = apply_merge(words, pair)
        learned_merges.append(pair)

        # Print required information
        print(
            f"Step {step:02d}: top pair = {pair} (count={count}) "
            f"-> vocab size = {vocab_size(words)}"
        )

    return learned_merges


# ----------------------------
# Segment a new word using learned merges
# ----------------------------
def segment(word, merges):
    tokens = tok_word(word)
    for pair in merges:
        tokens = merge_word(tokens, pair)
    return tokens


# ============================
# Main execution
# ============================
if __name__ == "__main__":

    # Toy corpus from Q2.1
    corpus = (
        "low low low low low lowest lowest "
        "newer newer newer newer newer newer "
        "wider wider wider new new"
    )

    # Train BPE
    merges = learn_bpe(corpus, max_merges=30)

    print("\nSegmentation results:")
    test_words = ["new", "newer", "lowest", "widest", "newestest"]
    for w in test_words:
        print(f"{w:<10} -> {' '.join(segment(w, merges))}")


Step 01: top pair = ('e', 'r') (count=9) -> vocab size = 11
Step 02: top pair = ('er', '_') (count=9) -> vocab size = 11
Step 03: top pair = ('n', 'e') (count=8) -> vocab size = 11
Step 04: top pair = ('ne', 'w') (count=8) -> vocab size = 11
Step 05: top pair = ('l', 'o') (count=7) -> vocab size = 10
Step 06: top pair = ('lo', 'w') (count=7) -> vocab size = 10
Step 07: top pair = ('new', 'er_') (count=6) -> vocab size = 11
Step 08: top pair = ('low', '_') (count=5) -> vocab size = 12
Step 09: top pair = ('w', 'i') (count=3) -> vocab size = 11
Step 10: top pair = ('wi', 'd') (count=3) -> vocab size = 10
Step 11: top pair = ('wid', 'er_') (count=3) -> vocab size = 9
Step 12: top pair = ('low', 'e') (count=2) -> vocab size = 8
Step 13: top pair = ('lowe', 's') (count=2) -> vocab size = 7
Step 14: top pair = ('lowes', 't') (count=2) -> vocab size = 6
Step 15: top pair = ('lowest', '_') (count=2) -> vocab size = 6
Step 16: top pair = ('new', '_') (count=2) -> vocab size = 5
Step 17: No bigr